# Week 12 Lab — Big Data Tools: SQL · APIs · PySpark
**Name:** ______  **Student ID:** ______  **Program:** ______

Each cell is self-contained: it has its own imports and loads/initializes its own data.
Colab: run `!pip install pyspark` first (Parts 3 and Bonus). Keep `week12_transactions.csv` and `week12_api_sample.json` next to this notebook.

In [1]:
!pip install pyspark -q

## Part 1 · SQL with Python
### Exercise 1.1(a) — 5 largest `send` transactions above 100,000 RWF

In [2]:
import sqlite3
import pandas as pd

# Initialize: load CSV and build the SQLite database
df = pd.read_csv('week12_transactions.csv')
conn = sqlite3.connect('momo.db')
df.to_sql('transactions', conn, if_exists='replace', index=False)

q = '''
SELECT txn_id, district, amount_rwf
FROM transactions
WHERE txn_type = 'send' AND amount_rwf > 100000
ORDER BY amount_rwf DESC
LIMIT 5
'''
pd.read_sql(q, conn)

,txn_id,district,amount_rwf
0,T01708,Kicukiro,140000
1,T00364,Huye,139000
2,T00895,Muhanga,139000
3,T00899,Nyarugenge,139000
4,T01314,Gasabo,139000


### Exercise 1.1(b) — Count flagged transactions (checkpoint: 11)

In [3]:
import sqlite3
import pandas as pd

# Initialize: load CSV and build the SQLite database
df = pd.read_csv('week12_transactions.csv')
conn = sqlite3.connect('momo.db')
df.to_sql('transactions', conn, if_exists='replace', index=False)

pd.read_sql('SELECT COUNT(*) AS flagged_count FROM transactions WHERE flagged = 1', conn)

,flagged_count
0,11


### Exercise 1.2(a) — Per `txn_type`: COUNT, SUM, AVG, ordered by total

In [4]:
import sqlite3
import pandas as pd

# Initialize: load CSV and build the SQLite database
df = pd.read_csv('week12_transactions.csv')
conn = sqlite3.connect('momo.db')
df.to_sql('transactions', conn, if_exists='replace', index=False)

q = '''
SELECT txn_type,
       COUNT(*)        AS n_txn,
       SUM(amount_rwf) AS total_rwf,
       AVG(amount_rwf) AS avg_rwf
FROM transactions
GROUP BY txn_type
ORDER BY total_rwf DESC
'''
res = pd.read_sql(q, conn)
print(res)
print('Type moving the most money:', res.loc[0, 'txn_type'])  # send ~62.0M

  txn_type  n_txn  total_rwf       avg_rwf
0     send    868   62049000  71485.023041
1  paybill    330   29970000  90818.181818
2  receive    489   27024000  55263.803681
3  airtime    313     833000   2661.341853
Type moving the most money: send


### Exercise 1.2(b) — Total per district

In [5]:
import sqlite3
import pandas as pd

# Initialize: load CSV and build the SQLite database
df = pd.read_csv('week12_transactions.csv')
conn = sqlite3.connect('momo.db')
df.to_sql('transactions', conn, if_exists='replace', index=False)

q = '''
SELECT district, SUM(amount_rwf) AS total_rwf
FROM transactions
GROUP BY district
ORDER BY total_rwf DESC
'''
res = pd.read_sql(q, conn)
print(res)
print('Leading district:', res.loc[0, 'district'])  # Muhanga ~21.1M

     district  total_rwf
0     Muhanga   21076000
1      Gasabo   20738000
2        Huye   18736000
3  Nyarugenge   18091000
4    Kicukiro   11338000
5     Musanze   10490500
6      Rubavu   10110500
7   Nyagatare    9296000
Leading district: Muhanga


### Exercise 1.3(a) — Transactions per hour in SQL, busiest first

In [6]:
import sqlite3
import pandas as pd

# Initialize: load CSV and build the SQLite database
df = pd.read_csv('week12_transactions.csv')
conn = sqlite3.connect('momo.db')
df.to_sql('transactions', conn, if_exists='replace', index=False)

q = '''
SELECT hour, COUNT(*) AS n_txn
FROM transactions
GROUP BY hour
ORDER BY n_txn DESC
'''
sql_hours = pd.read_sql(q, conn)
sql_hours.head(5)  # hour 20 with 185

,hour,n_txn
0,20,185
1,17,183
2,18,176
3,12,170
4,11,169


### Exercise 1.3(b) — Same question in Pandas, confirm both agree

In [7]:
import sqlite3
import pandas as pd

# Initialize: load CSV and build the SQLite database
df = pd.read_csv('week12_transactions.csv')
conn = sqlite3.connect('momo.db')
df.to_sql('transactions', conn, if_exists='replace', index=False)

sql_hours = pd.read_sql('SELECT hour, COUNT(*) AS n_txn FROM transactions GROUP BY hour ORDER BY n_txn DESC', conn)

pd_hours = df['hour'].value_counts()
print('Pandas busiest hour:', pd_hours.idxmax(), 'with', pd_hours.max())
print('SQL busiest hour:   ', sql_hours.loc[0, 'hour'], 'with', sql_hours.loc[0, 'n_txn'])
print('Agree:', pd_hours.idxmax() == sql_hours.loc[0, 'hour'] and pd_hours.max() == sql_hours.loc[0, 'n_txn'])

Pandas busiest hour: 20 with 185
SQL busiest hour:    20 with 185
Agree: True


## Part 2 · APIs & JSON
### Exercise 2.1 — JSON by hand
*Replace `json_string` with the one given in your lab notebook if it differs.*

In [8]:
import json

json_string = '{"city": "Kigali", "readings": [21.5, 23.0, 24.8, 26.1, 25.4]}'
obj = json.loads(json_string)

readings = obj['readings']
print('(a) City:', obj['city'])
print('(b) Number of readings:', len(readings))
print('(c) Average:', round(sum(readings) / len(readings), 2))

(a) City: Kigali
(b) Number of readings: 5
(c) Average: 24.16


### Exercise 2.2 — Call a real API (Open-Meteo, Kigali)

In [9]:
import json
import requests
import pandas as pd

# Initialize: call Open-Meteo for Kigali, fall back to the offline sample file
url = ('https://api.open-meteo.com/v1/forecast?latitude=-1.9441&longitude=30.0619'
       '&hourly=temperature_2m&forecast_days=1&timezone=Africa%2FKigali')
try:
    data = requests.get(url, timeout=10).json()
    assert 'hourly' in data
    print('Live API used')
except Exception:
    with open('week12_api_sample.json') as f:
        data = json.load(f)
    print('Offline sample used')
temps = data['hourly']['temperature_2m']
times = data['hourly']['time']

print('(a) Hours:', len(temps))
print('(b) Max:', max(temps), ' Min:', min(temps))
print('(c) Time of hottest hour:', times[temps.index(max(temps))])

Offline sample used
(a) Hours: 24
(b) Max: 27.9  Min: 15.9
(c) Time of hottest hour: 2026-11-22T14:00


### Exercise 2.3 — API data meets your toolbox

In [10]:
import json
import requests
import pandas as pd

# Initialize: call Open-Meteo for Kigali, fall back to the offline sample file
url = ('https://api.open-meteo.com/v1/forecast?latitude=-1.9441&longitude=30.0619'
       '&hourly=temperature_2m&forecast_days=1&timezone=Africa%2FKigali')
try:
    data = requests.get(url, timeout=10).json()
    assert 'hourly' in data
    print('Live API used')
except Exception:
    with open('week12_api_sample.json') as f:
        data = json.load(f)
    print('Offline sample used')
temps = data['hourly']['temperature_2m']
times = data['hourly']['time']

weather = pd.DataFrame({'time': times, 'temp_c': temps})
print('(a) describe():')
print(weather['temp_c'].describe())
mask = weather['temp_c'] >= 25
print('(b) Hours >= 25 C:', mask.sum())

Offline sample used
(a) describe():
count    24.000000
mean     21.137500
std       4.104696
min      15.900000
25%      17.650000
50%      19.900000
75%      25.025000
max      27.900000
Name: temp_c, dtype: float64
(b) Hours >= 25 C: 7


## Part 3 · PySpark
### Exercise 3.1 — Load & inspect

In [11]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialize: start Spark and load the CSV
spark = SparkSession.builder.master('local[*]').appName('Week12').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
sdf = spark.read.csv('week12_transactions.csv', header=True, inferSchema=True)

print('Row count:', sdf.count())
sdf.printSchema()
sdf.show(5)

Picked up JAVA_TOOL_OPTIONS: -XX:+IgnoreUnrecognizedVMOptions -XX:-UseBASE64Intrinsics
Picked up JAVA_TOOL_OPTIONS: -XX:+IgnoreUnrecognizedVMOptions -XX:-UseBASE64Intrinsics


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/23 12:14:20 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/11/23 12:14:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/usr/local/lib/python3.12/dist-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/11/23 12:14:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Row count: 2000
root
 |-- txn_id: string (nullable = true)
 |-- date: string (nullable = true)
 |-- hour: integer (nullable = true)
 |-- district: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount_rwf: integer (nullable = true)
 |-- flagged: integer (nullable = true)



+------+----------+----+----------+--------+----------+-------+
|txn_id|      date|hour|  district|txn_type|amount_rwf|flagged|
+------+----------+----+----------+--------+----------+-------+
|T00001| 11/1/2026|   8|    Gasabo|    send|      1000|      0|
|T00002| 11/9/2026|  12|Nyarugenge|    send|     25000|      0|
|T00003|11/19/2026|   8|   Muhanga|    send|     51000|      0|
|T00004| 11/1/2026|  17|      Huye|    send|     79000|      0|
|T00005|11/15/2026|  12|      Huye|    send|    109000|      0|
+------+----------+----+----------+--------+----------+-------+
only showing top 5 rows


### Exercise 3.2(a) — Transactions above 100,000 RWF (checkpoint: 451)

In [12]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialize: start Spark and load the CSV
spark = SparkSession.builder.master('local[*]').appName('Week12').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
sdf = spark.read.csv('week12_transactions.csv', header=True, inferSchema=True)

print('Above 100k:', sdf.filter(sdf.amount_rwf > 100000).count())

Above 100k: 451


### Exercise 3.2(b) — Show 5 flagged transactions

In [13]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialize: start Spark and load the CSV
spark = SparkSession.builder.master('local[*]').appName('Week12').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
sdf = spark.read.csv('week12_transactions.csv', header=True, inferSchema=True)

sdf.filter(sdf.flagged == 1).show(5)

+------+----------+----+--------+--------+----------+-------+
|txn_id|      date|hour|district|txn_type|amount_rwf|flagged|
+------+----------+----+--------+--------+----------+-------+
|T00279|11/29/2026|  19| Muhanga|    send|    127000|      1|
|T00326| 11/7/2026|  22|    Huye|    send|    127000|      1|
|T00536|11/27/2026|  17|  Gasabo|    send|    127000|      1|
|T00655|11/30/2026|  17|  Rubavu|    send|    134000|      1|
|T00697| 11/1/2026|   9| Muhanga|    send|    127000|      1|
+------+----------+----+--------+--------+----------+-------+
only showing top 5 rows


### Exercise 3.3(a) — Total amount per `txn_type`

In [14]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialize: start Spark and load the CSV
spark = SparkSession.builder.master('local[*]').appName('Week12').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
sdf = spark.read.csv('week12_transactions.csv', header=True, inferSchema=True)

sdf.groupBy('txn_type').sum('amount_rwf').show()

+--------+---------------+
|txn_type|sum(amount_rwf)|
+--------+---------------+
| paybill|       29970000|
| airtime|         833000|
| receive|       27024000|
|    send|       62049000|
+--------+---------------+



### Exercise 3.3(b) — Per-district totals with `.toPandas()`

In [15]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialize: start Spark and load the CSV
spark = SparkSession.builder.master('local[*]').appName('Week12').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
sdf = spark.read.csv('week12_transactions.csv', header=True, inferSchema=True)

district_totals = (sdf.groupBy('district').sum('amount_rwf')
                      .toPandas()
                      .sort_values('sum(amount_rwf)', ascending=False))
print(district_totals)

/usr/local/lib/python3.12/dist-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/dist-packages/pyspark/sql/pandas/conversion.py:374: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


     district  sum(amount_rwf)
4     Muhanga         21076000
5      Gasabo         20738000
6        Huye         18736000
1  Nyarugenge         18091000
0    Kicukiro         11338000
2     Musanze         10490500
7      Rubavu         10110500
3   Nyagatare          9296000


### Exercise 3.3(c) — Do Spark and SQL agree?

In [16]:
import sqlite3
import pandas as pd
from pyspark.sql import SparkSession

# Initialize both engines from the same CSV
df = pd.read_csv('week12_transactions.csv')
conn = sqlite3.connect('momo.db')
df.to_sql('transactions', conn, if_exists='replace', index=False)
spark = SparkSession.builder.master('local[*]').appName('Week12').getOrCreate()
sdf = spark.read.csv('week12_transactions.csv', header=True, inferSchema=True)

sql_type = pd.read_sql('SELECT txn_type, SUM(amount_rwf) AS t FROM transactions GROUP BY txn_type', conn).set_index('txn_type')['t'].sort_index()
sp_type = sdf.groupBy('txn_type').sum('amount_rwf').toPandas().set_index('txn_type')['sum(amount_rwf)'].sort_index()
sql_dist = pd.read_sql('SELECT district, SUM(amount_rwf) AS t FROM transactions GROUP BY district', conn).set_index('district')['t'].sort_index()
sp_dist = sdf.groupBy('district').sum('amount_rwf').toPandas().set_index('district')['sum(amount_rwf)'].sort_index()

# Comment: Yes - the Spark answers match the SQL answers exactly
# (send ~62.0M RWF; Muhanga leads with ~21.1M), because both run the same GROUP BY/SUM logic.
print('txn_type match:', (sql_type.values == sp_type.values).all())
print('district match:', (sql_dist.values == sp_dist.values).all())

/usr/local/lib/python3.12/dist-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/dist-packages/pyspark/sql/pandas/conversion.py:374: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.12/dist-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/dist-packages/pyspark/sql/pandas/conversion.py:374: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


txn_type match: True
district match: True


### Bonus (+3) — Spark speaks SQL too

In [17]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialize: start Spark and load the CSV
spark = SparkSession.builder.master('local[*]').appName('Week12').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
sdf = spark.read.csv('week12_transactions.csv', header=True, inferSchema=True)

sdf.createOrReplaceTempView('transactions')

spark.sql('''
    SELECT txn_type, COUNT(*) AS n_txn, SUM(amount_rwf) AS total_rwf, AVG(amount_rwf) AS avg_rwf
    FROM transactions
    GROUP BY txn_type
    ORDER BY total_rwf DESC
''').show()

# Comment: Same SQL on SQLite and on a thousand-machine cluster is powerful because you learn
# one language and reuse it at any scale: no rewriting code when data grows, Spark distributes
# the work automatically, and analysts, engineers and BI tools can all share the same queries.

+--------+-----+---------+------------------+
|txn_type|n_txn|total_rwf|           avg_rwf|
+--------+-----+---------+------------------+
|    send|  868| 62049000| 71485.02304147465|
| paybill|  330| 29970000| 90818.18181818182|
| receive|  489| 27024000|  55263.8036809816|
| airtime|  313|   833000|2661.3418530351437|
+--------+-----+---------+------------------+



## Part 4 · Choose Your Weapon
1. **SQL** — the data already lives in PostgreSQL, so a `GROUP BY` runs next to the data and returns only the small monthly-per-district result.
2. **Pandas** — 40,000 rows fit easily in memory, and Pandas plus matplotlib/Plotly and scikit-learn cover cleaning, charts and regression in one place.
3. **Spark** — 40 TB cannot fit on one machine, and Spark processes it in parallel where it already sits on the cluster.
4. **API** — only an API (e.g. Open-Meteo) delivers fresh data on demand, so the dashboard updates itself with no file uploads.

## Part 5 · Final-Project Proposal (Group D — Road Accident Severity and Risk in Kigali, Rwanda)
**Question:** What patterns (time of day, day of week, road conditions, vehicle type and location) are associated with road crashes in Kigali, and can police data predict whether a crash will be grievous or fatal?

**Dataset:** Kigali Traffic Police crash records for 2013 ("RW Police data 2013", hotspot_epi_rwanda_data_cleaned.csv), published openly and free on figshare (DOI 10.6084/m9.figshare.2058630) by Patel et al., BMC Public Health 2016, under CC BY 4.0 and de-identified. The paper reports 2,589 crashes (minimum 500 required) with four police severity levels, time of day, day of week, road surface and visibility, vehicle types and coordinates. We will confirm columns and record count when the file is loaded. If it proves unusable we will ask the lecturer to approve a clearly labeled alternative dataset.

**Why it matters:** Road crashes threaten public safety in Rwanda, and pedestrians, cyclists and motorcyclists suffer the most severe injuries. Knowing when, where and under which conditions serious crashes occur helps Rwanda National Police and Kigali authorities target enforcement, road design and awareness campaigns.

**Planned model:** Classification, because severity is a category (grievous or fatal versus non-grievous, about 18.5% positive). We will compare Logistic Regression, Decision Tree and Random Forest using a stratified train/test split, class weights for imbalance, and a confusion matrix, precision, recall and F1-score, then present results in an interactive Plotly HTML dashboard with limitations and recommendations.

*Full proposal: see the group's proposal document.*